# Databricks Lakehouse - End-to-End Pipeline Audit & Lineage Logging (`05_audit_logging`)

## Overview
This notebook provides end-to-end audit tracking across the Medallion Architecture (`Bronze` $\rightarrow$ `Silver` $\rightarrow$ `Gold`) within the `workspace.neuro_insight` schema. It generates a unique pipeline run ID, queries record counts across all lakehouse layers and quarantine tables, and inspects the underlying catalog metastore to verify table registrations.

---

### Step 1: Generate Pipeline Run Audit Metrics
Assign a unique execution UUID (`run_id`) and collect row counts across all medallion pipeline layers:
- **Bronze Layer**: Raw ingested telemetry data (`workspace.neuro_insight.bronze_telemetry`)[cite: 5].
- **Silver Layer**: Cleaned & validated patient metrics (`workspace.neuro_insight.silver_telemetry`)[cite: 5].
- **Quarantine**: Quarantined invalid/corrupt records (`workspace.neuro_insight.silver_quarantine`)[cite: 5].
- **Gold Layer**: Daily business aggregations (`workspace.neuro_insight.gold_daily_metrics`)[cite: 5].

In [ ]:
import uuid
from pyspark.sql import functions as F

# Generate unique execution UUID
run_id = str(uuid.uuid4())

# Extract row counts from Delta tables across medallion layers
bronze_count = spark.table("workspace.neuro_insight.bronze_telemetry").count()
silver_count = spark.table("workspace.neuro_insight.silver_telemetry").count()
quarantine_count = spark.table("workspace.neuro_insight.silver_quarantine").count()
gold_count = spark.table("workspace.neuro_insight.gold_daily_metrics").count()

# Print audit report
print("=== Pipeline Audit Run Summary ===")
print(f"Run ID           : {run_id}")
print(f"Bronze Records   : {bronze_count}")
print(f"Silver Records   : {silver_count}")
print(f"Quarantined Rows : {quarantine_count}")
print(f"Gold Aggregations: {gold_count}")

### Step 2: Metastore Table Inventory
Execute SQL catalog inspection to verify that all required Delta tables and log structures exist in the `workspace.neuro_insight` schema[cite: 5].

In [ ]:
%sql
-- Inspect all registered tables in the neuro_insight schema
SHOW TABLES IN workspace.neuro_insight;